# Configuracion comun

In [1]:
import sys
import os
from pathlib import PurePosixPath

current = os.getcwd()
etl_dir = None
for _ in range(5):
    candidate = os.path.join(current, "etl")
    if os.path.isdir(candidate):
        etl_dir = candidate
        break
    current = os.path.dirname(current)

if etl_dir is None:
    raise RuntimeError("No encontre la carpeta 'etl'. Revisar la estructura del proyecto.")

if etl_dir not in sys.path:
    sys.path.append(etl_dir)

print(f"etl encontrado en: {etl_dir}")

from pathlib import PurePosixPath
from extract.extract_annotations import list_xml_keys, download_xml
from load.load_to_s3 import copy_image, upload_bytes, download_text_if_exists

BUCKET = "humanoid-damage-cnn-834257582282-us-east-1-an"
PROCESSED_PREFIX = "processed/damage_classification/"
DRY_RUN = False  # cambiar a False cuando quieras subir de verdad

etl encontrado en: /home/sagemaker-user/humanoid-damage-cnn/etl


## Funciones compartidas de fusion y carga

Esto es lo unico que se repite igual para cualquier dataset, una vez que ya tenes la lista de objetos (tipo de dano + caja en pixeles). Se deja acá (no en un .py aparte) para que lo puedas ver y ajustar facil si hace falta.

In [2]:
def load_existing_classes(bucket, processed_prefix):
    """Lee el classes.txt actual (si existe) para no perder los IDs ya asignados."""
    content = download_text_if_exists(bucket, f"{processed_prefix}classes.txt")
    if content is None:
        return []
    return [line.strip() for line in content.splitlines() if line.strip()]


def merge_classes(existing_classes, new_damage_types):
    """Agrega al final los tipos de dano nuevos, sin tocar el orden/ID de los existentes."""
    merged = list(existing_classes)
    for damage_type in sorted(new_damage_types):
        if damage_type not in merged:
            merged.append(damage_type)
    return merged


def to_yolo_line(obj, class_to_id, width, height):
    """Convierte un objeto (dano con bndbox en pixeles) a una linea YOLO normalizada."""
    class_id = class_to_id[obj["damage_type"]]
    x_center = ((obj["xmin"] + obj["xmax"]) / 2) / width
    y_center = ((obj["ymin"] + obj["ymax"]) / 2) / height
    box_width = (obj["xmax"] - obj["xmin"]) / width
    box_height = (obj["ymax"] - obj["ymin"]) / height
    return f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"


def run_classification_etl(bucket, raw_annotations_prefix, raw_images_prefix, processed_prefix, parse_fn, dry_run=True):
    """Corre extract -> transform -> load para un dataset, fusionando con lo que ya haya en processed_prefix.
    parse_fn(xml_bytes) debe devolver: {"image_basename": str, "width": int, "height": int,
        "objects": [{"damage_type": str, "xmin": float, "ymin": float, "xmax": float, "ymax": float}, ...]}"""

    # --- Extract ---
    xml_keys = list_xml_keys(bucket, raw_annotations_prefix)
    print(f"Encontrados {len(xml_keys)} archivos XML en {raw_annotations_prefix}")

    # --- Transform ---
    parsed = []
    for key in xml_keys:
        try:
            xml_bytes = download_xml(bucket, key)
            annotation = parse_fn(xml_bytes)
        except Exception as exc:
            print(f"[ERROR] {key}: {exc}")
            continue
        if not annotation.get("image_basename"):
            print(f"[ERROR] {key}: no se pudo determinar el nombre de la imagen")
            continue
        if not annotation.get("objects"):
            continue  # sin daños anotados, no aporta a este dataset
        parsed.append(annotation)

    print(f"Anotaciones con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"

        dest_image_key = f"{processed_prefix}images/{basename}"
        copy_image(bucket, source_key, dest_image_key, dry_run=False)

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 50 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print("Listo.")
    return {"parsed": parsed, "all_classes": all_classes}

---
## Dataset `raw/IDEA/damages/`

Formato: un XML por imagen, tipo Pascal VOC (`<object><name damage="...">` + `<bndbox>`). El parser es especifico de este dataset, por eso vive acá y no en el bloque compartido de arriba.

In [3]:
import xml.etree.ElementTree as ET

def parse_dataset_1(xml_bytes):
    root = ET.fromstring(xml_bytes)

    filename_tag = root.findtext("filename", default="").strip()
    image_basename = PurePosixPath(filename_tag).name if filename_tag else None

    size_elem = root.find("size")
    width = int(size_elem.findtext("width")) if size_elem is not None else None
    height = int(size_elem.findtext("height")) if size_elem is not None else None

    objects = []
    for obj in root.findall("object"):
        name_elem = obj.find("name")
        if name_elem is None:
            continue
        damage_type = name_elem.attrib.get("damage") or (name_elem.text or "").strip()
        if not damage_type:
            continue
        bndbox = obj.find("bndbox")
        if bndbox is None:
            continue
        objects.append({
            "damage_type": damage_type.strip(),
            "xmin": float(bndbox.findtext("xmin")),
            "ymin": float(bndbox.findtext("ymin")),
            "xmax": float(bndbox.findtext("xmax")),
            "ymax": float(bndbox.findtext("ymax")),
        })

    return {"image_basename": image_basename, "width": width, "height": height, "objects": objects}

In [4]:
RAW_ANNOTATIONS_PREFIX_1 = "raw/IDEA/damages/annotations/"
RAW_IMAGES_PREFIX_1 = "raw/IDEA/damages/images/"

resultado_1 = run_classification_etl(
    bucket=BUCKET,
    raw_annotations_prefix=RAW_ANNOTATIONS_PREFIX_1,
    raw_images_prefix=RAW_IMAGES_PREFIX_1,
    processed_prefix=PROCESSED_PREFIX,
    parse_fn=parse_dataset_1,
    dry_run=DRY_RUN,
)

Encontrados 1727 archivos XML en raw/IDEA/damages/annotations/
Anotaciones con objetos: 1727
Tipos de dano nuevos agregados a classes.txt: ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar buckling', 'Rebar deformation', 'Reinforcement oxidized/corroded', 'Shear crack', 'Sliding shear crack', 'Stirrups break']
Total de clases acumuladas: 23 -> ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar bu

---
## Dataset `raw/Structural Defects 2.v1i.coco/damages/`

Formato: **COCO** — un solo JSON con `categories` (tipos de dano), `images` (metadata) y `annotations` (cajas, cada una con `image_id` apuntando a una imagen). El `bbox` viene como `[x_min, y_min, ancho, alto]` en pixeles (distinto al `xmin/ymin/xmax/ymax` de la Cajita 1), asi que hay que convertirlo.

Como es un solo archivo (no uno por imagen), la logica de extract/transform es distinta a la Cajita 1, pero el Load final usa las MISMAS funciones compartidas (`load_existing_classes`, `merge_classes`, `to_yolo_line`, `copy_image`, `upload_bytes`) para terminar en la misma carpeta.

In [11]:
import json

# Traduccion de los nombres de categoria (vienen en italiano en este dataset)
ITALIAN_TO_ENGLISH = {
    "Armatura corrosa": "Corroded reinforcement",
    "Corrosione": "Corrosion",
    "Distacco del copriferro": "Concrete cover detachment",
    "Fessura diagonale": "Diagonal crack",
}

def run_classification_etl_coco(bucket, annotation_key, raw_images_prefix, processed_prefix, dry_run=True):
    """Version de run_classification_etl para un dataset en formato COCO (un solo JSON para todas las imagenes)."""

    # --- Extract ---
    raw_bytes = download_xml(bucket, annotation_key)  # funciona igual, solo descarga bytes
    coco = json.loads(raw_bytes)
    print(f"JSON descargado: {len(coco['images'])} imagenes, {len(coco['annotations'])} anotaciones")

    # --- Transform ---
    category_id_to_name = {}
    for c in coco["categories"]:
        nombre_original = c["name"]
        nombre_en_ingles = ITALIAN_TO_ENGLISH.get(nombre_original, nombre_original)
        if nombre_original not in ITALIAN_TO_ENGLISH and nombre_original != "concrete-5ILF":
            print(f"[AVISO] '{nombre_original}' no esta en el diccionario de traduccion, se deja tal cual.")
        category_id_to_name[c["id"]] = nombre_en_ingles
    images_by_id = {img["id"]: img for img in coco["images"]}

    objects_by_image_id = {}
    for ann in coco["annotations"]:
        image_id = ann["image_id"]
        x_min, y_min, box_w, box_h = ann["bbox"]
        obj = {
            "damage_type": category_id_to_name[ann["category_id"]],
            "xmin": float(x_min),
            "ymin": float(y_min),
            "xmax": float(x_min) + float(box_w),
            "ymax": float(y_min) + float(box_h),
        }
        objects_by_image_id.setdefault(image_id, []).append(obj)

    parsed = []
    for image_id, objects in objects_by_image_id.items():
        img = images_by_id.get(image_id)
        if img is None:
            print(f"[ERROR] image_id {image_id} no encontrado en 'images'")
            continue
        parsed.append({
            "image_basename": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "objects": objects,
        })

    print(f"Imagenes con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (identico a la Cajita 1) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"

        dest_image_key = f"{processed_prefix}images/{basename}"
        copy_image(bucket, source_key, dest_image_key, dry_run=False)

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print("Listo.")
    return {"parsed": parsed, "all_classes": all_classes}

In [12]:
ANNOTATION_KEY_2 = "raw/Structural Defects 2.v1i.coco/damages/annotations/_annotations.coco.json"
RAW_IMAGES_PREFIX_2 = "raw/Structural Defects 2.v1i.coco/damages/images/"

resultado_2 = run_classification_etl_coco(
    bucket=BUCKET,
    annotation_key=ANNOTATION_KEY_2,
    raw_images_prefix=RAW_IMAGES_PREFIX_2,
    processed_prefix=PROCESSED_PREFIX,
    dry_run=DRY_RUN,
)

JSON descargado: 6762 imagenes, 10689 anotaciones
Imagenes con objetos: 4726
Tipos de dano nuevos agregados a classes.txt: ['Concrete cover detachment', 'Corrosion', 'Diagonal crack']
Total de clases acumuladas: 26 -> ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar buckling', 'Rebar deformation', 'Reinforcement oxidized/corroded', 'Shear crack', 'Sliding shear crack', 'Stirrups break', 'Concrete cover detachment', 'Corrosion', 'Diagonal crack']
Procesadas 200/4726 imagenes...
Procesadas 400/4726 imagenes...
Procesadas 600/4726 imagenes...
Procesadas 800/4726 imagenes...
Procesadas 1000/4726 imagenes...
Procesadas 1200/4726 imagenes...
Procesadas 1400/4726 imagenes...
Procesadas 1600/4726 

In [3]:
---
## Dataset `raw/18k_varias_clasesyraras_roboflow/`

Formato: **COCO**, repartido en 3 archivos JSON (`_annotations.coco1.json`, `_annotations.coco2.json`, `_annotations.coco3.json`), cada uno con sus propias `categories`, `images` y `annotations`. La categoria id 0 es la supercategoria raiz (`Deterioration-Object-Detect-mWSG`), no un tipo de dano real, asi que se excluye al parsear. El `bbox` viene como `[x_min, y_min, ancho, alto]` en pixeles, igual que en la Cajita 2, y se convierte de la misma forma.

Todas las imagenes viven juntas en una sola carpeta `images/` (sin separacion train/valid/test). El Load final usa las MISMAS funciones compartidas (`load_existing_classes`, `merge_classes`, `to_yolo_line`, `copy_image`, `upload_bytes`) para terminar en la misma carpeta `processed/`.

SyntaxError: invalid syntax (1961709368.py, line 1)

In [19]:
import json
from botocore.exceptions import ClientError

def parse_coco_split(coco_bytes):
    """Parsea un JSON COCO de un split y devuelve la lista de anotaciones (sin la categoria raiz)."""
    coco = json.loads(coco_bytes)

    category_id_to_name = {
        c["id"]: c["name"]
        for c in coco["categories"]
        if c.get("supercategory") != "none"  # descarta la categoria raiz
    }
    images_by_id = {img["id"]: img for img in coco["images"]}

    objects_by_image_id = {}
    for ann in coco["annotations"]:
        category_id = ann["category_id"]
        if category_id not in category_id_to_name:
            continue  # anotacion perteneciente a la categoria raiz, se ignora
        image_id = ann["image_id"]
        x_min, y_min, box_w, box_h = ann["bbox"]
        obj = {
            "damage_type": category_id_to_name[category_id],
            "xmin": float(x_min),
            "ymin": float(y_min),
            "xmax": float(x_min) + float(box_w),
            "ymax": float(y_min) + float(box_h),
        }
        objects_by_image_id.setdefault(image_id, []).append(obj)

    parsed = []
    for image_id, objects in objects_by_image_id.items():
        img = images_by_id.get(image_id)
        if img is None:
            print(f"[ERROR] image_id {image_id} no encontrado en 'images'")
            continue
        parsed.append({
            "image_basename": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "objects": objects,
        })

    return parsed


def run_classification_etl_coco_multi(bucket, annotation_keys, raw_images_prefix, processed_prefix, dry_run=True):
    """Version de run_classification_etl para un dataset COCO repartido en varios archivos JSON."""

    # --- Extract + Transform ---
    parsed = []
    for key in annotation_keys:
        raw_bytes = download_xml(bucket, key)
        split_parsed = parse_coco_split(raw_bytes)
        print(f"{key}: {len(split_parsed)} imagenes con objetos")
        parsed.extend(split_parsed)

    print(f"Total imagenes con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (identico a los datasets anteriores, salteando imagenes faltantes) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    omitidas = 0
    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"
        dest_image_key = f"{processed_prefix}images/{basename}"

        try:
            copy_image(bucket, source_key, dest_image_key, dry_run=False)
        except ClientError as exc:
            print(f"[ERROR] no existe la imagen {source_key}, se omite")
            omitidas += 1
            continue

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print(f"Listo. Imagenes omitidas por no encontrarse en {raw_images_prefix}: {omitidas}")
    return {"parsed": parsed, "all_classes": all_classes}

In [12]:
RAW_ANNOTATIONS_PREFIX_3 = "raw/18k_varias_clasesyraras_roboflow/annotations/"
RAW_IMAGES_PREFIX_3 = "raw/18k_varias_clasesyraras_roboflow/images/"

ANNOTATION_KEYS_3 = [
    f"{RAW_ANNOTATIONS_PREFIX_3}_annotations.coco1.json",
    f"{RAW_ANNOTATIONS_PREFIX_3}_annotations.coco2.json",
    f"{RAW_ANNOTATIONS_PREFIX_3}_annotations.coco3.json",
]

resultado_3 = run_classification_etl_coco_multi(
    bucket=BUCKET,
    annotation_keys=ANNOTATION_KEYS_3,
    raw_images_prefix=RAW_IMAGES_PREFIX_3,
    processed_prefix=PROCESSED_PREFIX,
    dry_run=DRY_RUN,
)

raw/18k_varias_clasesyraras_roboflow/annotations/_annotations.coco1.json: 3626 imagenes con objetos
raw/18k_varias_clasesyraras_roboflow/annotations/_annotations.coco2.json: 12690 imagenes con objetos
raw/18k_varias_clasesyraras_roboflow/annotations/_annotations.coco3.json: 1812 imagenes con objetos
Total imagenes con objetos: 18128
Total de clases acumuladas: 28 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread']
Procesadas 200/18128 imagen

In [ ]:
---
## Dataset `raw/5_clases_distintas_roboflow/`

Formato: **COCO**, repartido en 3 archivos JSON (`_annotations.coco.json`, `_annotations.coco1.json`, `_annotations.coco2.json`), mismo patron que el dataset de 18k. La categoria id 0 es la supercategoria raiz (`building-damage-insurance`), se excluye al parsear. El `bbox` viene como `[x_min, y_min, ancho, alto]` en pixeles, igual que en las Cajitas anteriores.

Todas las imagenes viven juntas en una sola carpeta `images/`. Reutiliza las mismas funciones `parse_coco_split` y `run_classification_etl_coco_multi` del dataset de 18k. El Load final usa las MISMAS funciones compartidas (`load_existing_classes`, `merge_classes`, `to_yolo_line`, `copy_image`, `upload_bytes`) para terminar en la misma carpeta `processed/`.

In [11]:
RAW_ANNOTATIONS_PREFIX_4 = "raw/5_clases_distintas_roboflow/annotations/"
RAW_IMAGES_PREFIX_4 = "raw/5_clases_distintas_roboflow/images/"

ANNOTATION_KEYS_4 = [
    f"{RAW_ANNOTATIONS_PREFIX_4}_annotations.coco.json",
    f"{RAW_ANNOTATIONS_PREFIX_4}_annotations.coco1.json",
    f"{RAW_ANNOTATIONS_PREFIX_4}_annotations.coco2.json",
]

resultado_4 = run_classification_etl_coco_multi(
    bucket=BUCKET,
    annotation_keys=ANNOTATION_KEYS_4,
    raw_images_prefix=RAW_IMAGES_PREFIX_4,
    processed_prefix=PROCESSED_PREFIX,
    dry_run=DRY_RUN,
)

raw/5_clases_distintas_roboflow/annotations/_annotations.coco.json: 4228 imagenes con objetos
raw/5_clases_distintas_roboflow/annotations/_annotations.coco1.json: 547 imagenes con objetos
raw/5_clases_distintas_roboflow/annotations/_annotations.coco2.json: 348 imagenes con objetos
Total imagenes con objetos: 5123
Tipos de dano nuevos agregados a classes.txt: ['0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage']
Total de clases acumuladas: 38 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumblin

In [ ]:
---
## Dataset `raw/corrosion & spalling_kaggle/`

Formato distinto a los anteriores: NO hay archivo de annotations ni bounding boxes, es un dataset de clasificacion. Todas las imagenes (excepto las que terminan en `_lab`, que son mascaras y se excluyen) representan AMBOS danos a la vez: `Corrosion` y `Concrete spalling` (nombres ya existentes en `classes.txt`). Como no hay bbox real, se usa la imagen completa como caja para cada dano presente.

El Load final usa las MISMAS funciones compartidas (`load_existing_classes`, `merge_classes`, `to_yolo_line`, `copy_image`, `upload_bytes`).

In [6]:
from PIL import Image
import io

from extract.s3_utils import listar_imagenes, descargar_bytes

def is_lab_image(basename):
    """Las imagenes que terminan en '_lab' antes de la extension son mascaras, se excluyen."""
    stem = PurePosixPath(basename).stem
    return stem.endswith("_lab")


def run_classification_etl_whole_image(bucket, raw_images_prefix, processed_prefix, damage_types, dry_run=True):
    """
    ETL para datasets de clasificacion (sin bboxes ni archivo de annotations),
    donde cada imagen (que no sea '_lab') tiene TODOS los damage_types indicados
    presentes en la imagen completa.
    """

    # --- Extract ---
    all_keys = listar_imagenes(bucket, raw_images_prefix)
    image_keys = [k for k in all_keys if not is_lab_image(k)]
    lab_keys = [k for k in all_keys if is_lab_image(k)]
    print(f"Imagenes totales: {len(all_keys)} | validas: {len(image_keys)} | excluidas (_lab): {len(lab_keys)}")

    # --- Transform ---
    parsed = []
    for key in image_keys:
        basename = PurePosixPath(key).name
        try:
            img_bytes = descargar_bytes(bucket, key)
            with Image.open(io.BytesIO(img_bytes)) as img:
                width, height = img.size
        except Exception as exc:
            print(f"[ERROR] no se pudo abrir {key}, se omite: {exc}")
            continue

        objects = [
            {
                "damage_type": damage_type,
                "xmin": 0.0,
                "ymin": 0.0,
                "xmax": float(width),
                "ymax": float(height),
            }
            for damage_type in damage_types
        ]
        parsed.append({
            "image_basename": basename,
            "width": width,
            "height": height,
            "objects": objects,
        })

    print(f"Total imagenes procesadas: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    else:
        print("No hay tipos de dano nuevos, ya existian en classes.txt")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (mismas funciones compartidas) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    omitidas = 0
    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"
        dest_image_key = f"{processed_prefix}images/{basename}"

        try:
            copy_image(bucket, source_key, dest_image_key, dry_run=False)
        except ClientError:
            print(f"[ERROR] no existe la imagen {source_key}, se omite")
            omitidas += 1
            continue

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print(f"Listo. Imagenes omitidas por no encontrarse en {raw_images_prefix}: {omitidas}")
    return {"parsed": parsed, "all_classes": all_classes}

In [23]:
RAW_IMAGES_PREFIX_5 = "raw/corrosion & spalling_kaggle/"

resultado_5 = run_classification_etl_whole_image(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_5,
    processed_prefix=PROCESSED_PREFIX,
    damage_types=["Corrosion", "Concrete spalling"],
    dry_run=DRY_RUN,
)

Imagenes totales: 3160 | validas: 1580 | excluidas (_lab): 1580
Total imagenes procesadas: 1580
No hay tipos de dano nuevos, ya existian en classes.txt
Total de clases acumuladas: 38 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage']
Procesadas 200/1580 imagenes...
Procesadas 400/1580 imagenes...
Procesadas 600/1580 imagenes...
Proce

In [ ]:
---
## Dataset `raw/corrosion_roboflow/images/`

Formato: **COCO**, un solo archivo `_annotations.coco.json`, ubicado dentro de la misma carpeta `images/` (no en una carpeta `annotations/` separada). La categoria id 0 es la supercategoria raiz (`supercategory: "none"`), se excluye. El resto de las categorias son subtipos de corrosion (ej. `Crevice-Corrosion`); todas se mapean a una unica clase `Corrosion`. El `bbox` viene como `[x_min, y_min, ancho, alto]` en pixeles, igual que en las Cajitas anteriores.

In [7]:
def run_classification_etl_coco_single_mapped(bucket, annotation_key, raw_images_prefix, processed_prefix, category_mapping_fn, dry_run=True):
    """
    Version de run_classification_etl_coco para un dataset COCO de un solo archivo,
    donde ademas se remapean los nombres de categoria via category_mapping_fn
    (por ejemplo, para colapsar varios subtipos en una sola clase).
    Excluye la categoria raiz (supercategory == "none").
    """

    # --- Extract ---
    raw_bytes = download_xml(bucket, annotation_key)
    coco = json.loads(raw_bytes)
    print(f"JSON descargado: {len(coco['images'])} imagenes, {len(coco['annotations'])} anotaciones")

    # --- Transform ---
    category_id_to_name = {
        c["id"]: category_mapping_fn(c["name"])
        for c in coco["categories"]
        if c.get("supercategory") != "none"
    }
    images_by_id = {img["id"]: img for img in coco["images"]}

    objects_by_image_id = {}
    for ann in coco["annotations"]:
        category_id = ann["category_id"]
        if category_id not in category_id_to_name:
            continue
        image_id = ann["image_id"]
        x_min, y_min, box_w, box_h = ann["bbox"]
        obj = {
            "damage_type": category_id_to_name[category_id],
            "xmin": float(x_min),
            "ymin": float(y_min),
            "xmax": float(x_min) + float(box_w),
            "ymax": float(y_min) + float(box_h),
        }
        objects_by_image_id.setdefault(image_id, []).append(obj)

    parsed = []
    for image_id, objects in objects_by_image_id.items():
        img = images_by_id.get(image_id)
        if img is None:
            print(f"[ERROR] image_id {image_id} no encontrado en 'images'")
            continue
        parsed.append({
            "image_basename": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "objects": objects,
        })

    print(f"Imagenes con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    else:
        print("No hay tipos de dano nuevos, ya existian en classes.txt")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (mismas funciones compartidas) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    omitidas = 0
    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"
        dest_image_key = f"{processed_prefix}images/{basename}"

        try:
            copy_image(bucket, source_key, dest_image_key, dry_run=False)
        except ClientError:
            print(f"[ERROR] no existe la imagen {source_key}, se omite")
            omitidas += 1
            continue

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print(f"Listo. Imagenes omitidas por no encontrarse en {raw_images_prefix}: {omitidas}")
    return {"parsed": parsed, "all_classes": all_classes}

In [25]:
RAW_ANNOTATIONS_KEY_6 = "raw/corrosion_roboflow/images/_annotations.coco.json"
RAW_IMAGES_PREFIX_6 = "raw/corrosion_roboflow/images/"

resultado_6 = run_classification_etl_coco_single_mapped(
    bucket=BUCKET,
    annotation_key=RAW_ANNOTATIONS_KEY_6,
    raw_images_prefix=RAW_IMAGES_PREFIX_6,
    processed_prefix=PROCESSED_PREFIX,
    category_mapping_fn=lambda name: "Corrosion",
    dry_run=DRY_RUN,
)

JSON descargado: 4949 imagenes, 20007 anotaciones
Imagenes con objetos: 4949
No hay tipos de dano nuevos, ya existian en classes.txt
Total de clases acumuladas: 38 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage']
Procesadas 200/4949 imagenes...
Procesadas 400/4949 imagenes...
Procesadas 600/4949 imagenes...
Procesadas 800/4949 imag

In [3]:
---
## Dataset `raw/efflorescence_zenodo/`

Formato: sin annotations, dataset de clasificacion. Todas las imagenes en `images/` corresponden al dano `Efflorescence`. Se usa la imagen completa como caja (igual que corrosion & spalling). Reutiliza `run_classification_etl_whole_image`.

SyntaxError: invalid syntax (31562472.py, line 1)

In [10]:
RAW_IMAGES_PREFIX_7 = "raw/efflorescence_zenodo/images/"

resultado_7 = run_classification_etl_whole_image(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_7,
    processed_prefix=PROCESSED_PREFIX,
    damage_types=["Efflorescence"],
    dry_run=DRY_RUN,
)

Imagenes totales: 100 | validas: 100 | excluidas (_lab): 0
Total imagenes procesadas: 100
Tipos de dano nuevos agregados a classes.txt: ['Efflorescence']
Total de clases acumuladas: 39 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage', 'Efflorescence']
Procesadas 100/100 imagenes...
Listo. Imagenes omitidas por no encontrarse en raw/

In [ ]:
---
## Dataset `raw/MUCHAS CLASES_roboflow/`

Formato: **COCO**, repartido en 3 archivos JSON (`_annotations.coco1.json`, `_annotations.coco2.json`, `_annotations.coco3.json`) en la carpeta `annotations/`, mismo patron que 18k y 5_clases. La categoria id 0 es la supercategoria raiz (`Suggaring-Detection-Via-Sem-o0v0`), se excluye al parsear. Imagenes todas juntas en `images/`. Reutiliza `parse_coco_split` y `run_classification_etl_coco_multi`.

In [22]:
RAW_ANNOTATIONS_PREFIX_8 = "raw/MUCHAS CLASES_roboflow/annotations/"
RAW_IMAGES_PREFIX_8 = "raw/MUCHAS CLASES_roboflow/"

ANNOTATION_KEYS_8 = [
    f"{RAW_ANNOTATIONS_PREFIX_8}_annotations.coco1.json",
    f"{RAW_ANNOTATIONS_PREFIX_8}_annotations.coco2.json",
    f"{RAW_ANNOTATIONS_PREFIX_8}_annotations.coco3.json",
]

resultado_8 = run_classification_etl_coco_multi(
    bucket=BUCKET,
    annotation_keys=ANNOTATION_KEYS_8,
    raw_images_prefix=RAW_IMAGES_PREFIX_8,
    processed_prefix=PROCESSED_PREFIX,
    dry_run=DRY_RUN,
)

raw/MUCHAS CLASES_roboflow/annotations/_annotations.coco1.json: 863 imagenes con objetos
raw/MUCHAS CLASES_roboflow/annotations/_annotations.coco2.json: 1725 imagenes con objetos
raw/MUCHAS CLASES_roboflow/annotations/_annotations.coco3.json: 6038 imagenes con objetos
Total imagenes con objetos: 8626
Total de clases acumuladas: 64 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_pain

In [ ]:
---
## Dataset `raw/soil_moisture_mendeley/`

Formato: sin annotations, dataset de clasificacion. Todas las imagenes en `images/` corresponden al dano `Soil moisture`. Se usa la imagen completa como caja. Reutiliza `run_classification_etl_whole_image`.

In [12]:
RAW_IMAGES_PREFIX_9 = "raw/soil_moisture_mendeley/images/"

resultado_9 = run_classification_etl_whole_image(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_9,
    processed_prefix=PROCESSED_PREFIX,
    damage_types=["Soil moisture"],
    dry_run=DRY_RUN,
)

Imagenes totales: 1177 | validas: 1177 | excluidas (_lab): 0
Total imagenes procesadas: 1177
Tipos de dano nuevos agregados a classes.txt: ['Soil moisture']
Total de clases acumuladas: 59 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage', 'Efflorescence', '-1.Crumbling', '-2.Corrosion-Local-Widespread', '-3.Delamination-Localized-Spr

In [ ]:
---
## Dataset `raw/vegetation_zenodo/`

Formato: sin annotations, dataset de clasificacion. Todas las imagenes estan directamente en `raw/vegetation_zenodo/` (sin subcarpeta `images/`) y corresponden al dano `Vegetation`. Se usa la imagen completa como caja. Reutiliza `run_classification_etl_whole_image`.

In [13]:
RAW_IMAGES_PREFIX_10 = "raw/vegetation_zenodo/"

resultado_10 = run_classification_etl_whole_image(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_10,
    processed_prefix=PROCESSED_PREFIX,
    damage_types=["Vegetation"],
    dry_run=DRY_RUN,
)

Imagenes totales: 4255 | validas: 4255 | excluidas (_lab): 0
Total imagenes procesadas: 4255
Tipos de dano nuevos agregados a classes.txt: ['Vegetation']
Total de clases acumuladas: 60 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage', 'Efflorescence', '-1.Crumbling', '-2.Corrosion-Local-Widespread', '-3.Delamination-Localized-Spread

In [ ]:
---
## Dataset `raw/void, spalling, honeycomb y crack_mendeley/images/`

Formato: sin annotations, dataset de clasificacion organizado en subcarpetas por clase dentro de `images/` (`crack/`, `Honeycomb/`, `Spalling/`, `VOID/`). Cada imagen pertenece unicamente al dano de su subcarpeta (mapeo: crack->Crack, Spalling->Concrete spalling, Honeycomb->Honeycomb, VOID->Void). Se usa la imagen completa como caja.

In [14]:
def run_classification_etl_folders(bucket, raw_images_prefix, processed_prefix, folder_to_damage_type, dry_run=True):
    """
    ETL para datasets de clasificacion organizados en subcarpetas por clase
    (sin annotations, sin bboxes). folder_to_damage_type mapea el nombre de
    la subcarpeta (tal cual aparece en S3) al nombre de clase de damage_type.
    """

    # --- Extract + Transform ---
    parsed = []
    for folder_name, damage_type in folder_to_damage_type.items():
        folder_prefix = f"{raw_images_prefix}{folder_name}/"
        keys = listar_imagenes(bucket, folder_prefix)
        print(f"{folder_prefix}: {len(keys)} imagenes -> {damage_type}")

        for key in keys:
            basename = PurePosixPath(key).name
            try:
                img_bytes = descargar_bytes(bucket, key)
                with Image.open(io.BytesIO(img_bytes)) as img:
                    width, height = img.size
            except Exception as exc:
                print(f"[ERROR] no se pudo abrir {key}, se omite: {exc}")
                continue

            parsed.append({
                "image_basename": basename,
                "source_key": key,
                "width": width,
                "height": height,
                "objects": [{
                    "damage_type": damage_type,
                    "xmin": 0.0,
                    "ymin": 0.0,
                    "xmax": float(width),
                    "ymax": float(height),
                }],
            })

    print(f"Total imagenes procesadas: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (mismas funciones compartidas) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    omitidas = 0
    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = annotation["source_key"]
        dest_image_key = f"{processed_prefix}images/{basename}"

        try:
            copy_image(bucket, source_key, dest_image_key, dry_run=False)
        except ClientError:
            print(f"[ERROR] no existe la imagen {source_key}, se omite")
            omitidas += 1
            continue

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print(f"Listo. Imagenes omitidas: {omitidas}")
    return {"parsed": parsed, "all_classes": all_classes}

In [15]:
RAW_IMAGES_PREFIX_11 = "raw/void, spalling, honeycomb y crack_mendeley/images/"

FOLDER_TO_DAMAGE_TYPE_11 = {
    "crack": "Crack",
    "Honeycomb": "Honeycomb",
    "Spalling": "Concrete spalling",
    "VOID": "Void",
}

resultado_11 = run_classification_etl_folders(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_11,
    processed_prefix=PROCESSED_PREFIX,
    folder_to_damage_type=FOLDER_TO_DAMAGE_TYPE_11,
    dry_run=DRY_RUN,
)

raw/void, spalling, honeycomb y crack_mendeley/images/crack/: 365 imagenes -> Crack
raw/void, spalling, honeycomb y crack_mendeley/images/Honeycomb/: 511 imagenes -> Honeycomb
raw/void, spalling, honeycomb y crack_mendeley/images/Spalling/: 278 imagenes -> Concrete spalling
raw/void, spalling, honeycomb y crack_mendeley/images/VOID/: 220 imagenes -> Void
Total imagenes procesadas: 1374
Tipos de dano nuevos agregados a classes.txt: ['Honeycomb', 'Void']
Total de clases acumuladas: 62 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corros

In [ ]:
---
## Dataset `raw/water_kaggle/`

Formato: **YOLO** (no COCO/Pascal VOC). Un archivo `.txt` por imagen en `annotations/`, mismo nombre base que la imagen, ya con bbox normalizado `class_id x_center y_center width height`. Mapeo local de clases: `0` -> `Stagnant water`, `1` -> `Wet surface`. Solo hace falta remapear el `class_id` local al id global fusionado en `classes.txt`, no hay que convertir coordenadas.

In [16]:
LOCAL_CLASS_MAP_12 = {
    "0": "Stagnant water",
    "1": "Wet surface",
}

def run_classification_etl_yolo(bucket, raw_images_prefix, raw_annotations_prefix, processed_prefix, local_class_map, dry_run=True):
    """
    ETL para datasets que ya vienen en formato YOLO (un .txt por imagen,
    class_id x_center y_center width height, todo normalizado 0-1).
    local_class_map mapea el class_id local (como string) al nombre real del dano.
    Solo se remapea el class_id, las coordenadas se copian tal cual.
    """

    # --- Extract ---
    image_keys = listar_imagenes(bucket, raw_images_prefix)
    print(f"Imagenes encontradas: {len(image_keys)}")

    # --- Transform ---
    parsed = []
    for key in image_keys:
        basename = PurePosixPath(key).name
        stem = PurePosixPath(key).stem
        label_key = f"{raw_annotations_prefix}{stem}.txt"

        try:
            label_bytes = download_xml(bucket, label_key)  # solo descarga bytes, sirve para cualquier texto
        except ClientError:
            print(f"[ERROR] no existe el annotation {label_key}, se omite")
            continue

        lines = label_bytes.decode("utf-8").strip().splitlines()
        objects = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            local_id, x_c, y_c, w, h = parts[0], parts[1], parts[2], parts[3], parts[4]
            if local_id not in local_class_map:
                print(f"[AVISO] class_id local '{local_id}' no esta en local_class_map, se omite esa linea en {label_key}")
                continue
            objects.append({
                "damage_type": local_class_map[local_id],
                "x_center": float(x_c),
                "y_center": float(y_c),
                "width_norm": float(w),
                "height_norm": float(h),
            })

        parsed.append({
            "image_basename": basename,
            "source_key": key,
            "objects": objects,
        })

    print(f"Total imagenes procesadas: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    omitidas = 0
    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = annotation["source_key"]
        dest_image_key = f"{processed_prefix}images/{basename}"

        try:
            copy_image(bucket, source_key, dest_image_key, dry_run=False)
        except ClientError:
            print(f"[ERROR] no existe la imagen {source_key}, se omite")
            omitidas += 1
            continue

        label_lines = [
            f"{class_to_id[obj['damage_type']]} {obj['x_center']:.6f} {obj['y_center']:.6f} {obj['width_norm']:.6f} {obj['height_norm']:.6f}"
            for obj in annotation["objects"]
        ]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print(f"Listo. Imagenes omitidas: {omitidas}")
    return {"parsed": parsed, "all_classes": all_classes}

In [17]:
RAW_IMAGES_PREFIX_12 = "raw/water_kaggle/images/"
RAW_ANNOTATIONS_PREFIX_12 = "raw/water_kaggle/annotations/"

resultado_12 = run_classification_etl_yolo(
    bucket=BUCKET,
    raw_images_prefix=RAW_IMAGES_PREFIX_12,
    raw_annotations_prefix=RAW_ANNOTATIONS_PREFIX_12,
    processed_prefix=PROCESSED_PREFIX,
    local_class_map=LOCAL_CLASS_MAP_12,
    dry_run=DRY_RUN,
)

Imagenes encontradas: 2000
Total imagenes procesadas: 2000
Tipos de dano nuevos agregados a classes.txt: ['Stagnant water', 'Wet surface']
Total de clases acumuladas: 64 -> ['Concrete spalling', 'Corrosion', 'Crack', 'Debris', 'Detachment crack', 'Diagonal crack', 'Flexural/rocking crack', 'Large displacement', 'Leaching', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar deformation/buckling', 'Shear crack', '-1-Earthquake-Induced Cracks And Collapses', '-2-Blistering', '-3-Alveolization', '-4-Vegetation-Advanced Stage', '-5-Detachment', '-6-Pitting', '-7- Swelling', '-8-Flaking', '-9-Exfoliation', '10-Vandalism', '11-Crumbling', '12-Corrosion-Local Widespread', '13-Delamination-Localized Spread', '14-Encrustation-Localized Spread', '0', '1', 'crack', 'damp', 'dampness', 'mold', 'peeling_paint', 'stain', 'stairstep_crack', 'water_seepage', 'Efflorescence', '-1.Crumbling', '-2.Corrosion-Local-Widespread', '-3.Delamination-Localized-Spread', '-5-Salt Cry